In [13]:
#!pip install openpyxl

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import BertTokenizer
from torch.serialization import add_safe_globals

import sys
import os

import tkinter as tk
from tkinter import filedialog, ttk, messagebox
import threading
import queue
import re

In [15]:
class HierarchicalBertClassifier(nn.Module):
    def __init__(self, num_impactarea_labels, num_genome_labels, num_outcome_labels, hidden_size=256, dropout_rate=0.3):
        super(HierarchicalBertClassifier, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.dropout = nn.Dropout(dropout_rate)

        #impact area prediction
        self.ia_hidden = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.ia_classifier = nn.Linear(hidden_size, num_impactarea_labels)
        #layer to extract ia features
        self.ia_feature_extractor = nn.Linear(num_impactarea_labels, hidden_size)

        #genome prediction (conditioned on impact area)
        self.genome_hidden = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size + num_impactarea_labels, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.genome_classifier = nn.Linear(hidden_size, num_genome_labels)
        #layer to extract genome features
        self.genome_feature_extractor = nn.Linear(num_genome_labels, hidden_size)

        #outcome prediction (conditioned on impact area & genome)
        self.cross_attention = nn.MultiheadAttention(hidden_size, num_heads=4)
        self.outcome_hidden = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size + num_impactarea_labels + num_genome_labels + hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.outcome_classifier = nn.Linear(hidden_size, num_outcome_labels)


    def forward(self, input_ids, attention_mask):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        bert_output = self.dropout(bert_output)

        #impact area prediction
        ia_hidden = self.ia_hidden(bert_output)
        ia_logits = self.ia_classifier(ia_hidden)
        #get probs for ia to pass to next level
        ia_probs= F.softmax(ia_logits, dim=1)
        #extract more features from ia probs
        ia_features = self.ia_feature_extractor(ia_probs)

        #genome prediction (with impact area info)
        genome_input = torch.cat([bert_output, ia_probs], dim=1)
        genome_hidden = self.genome_hidden(genome_input)
        genome_logits = self.genome_classifier(genome_hidden)
        #get probs for genome to pass to next level
        genome_probs = F.softmax(genome_logits, dim=1)

        #extract more features from genome probs
        genome_features = self.genome_feature_extractor(genome_probs)

        #cross atttention between genome & ia features
        genome_features_reshaped=genome_features.unsqueeze(0)
        ia_features_reshaped= ia_features.unsqueeze(0)

        attended_features, _ = self.cross_attention(
        genome_features_reshaped,  # query
        ia_features_reshaped,      # key
        ia_features_reshaped       # value
       )
        attended_features = attended_features.squeeze(0)

        #outcome prediction
        outcome_input = torch.cat([bert_output, ia_probs, genome_probs, attended_features], dim=1)
        outcome_hidden = self.outcome_hidden(outcome_input)
        outcome_logits = self.outcome_classifier(outcome_hidden)

        return ia_logits, genome_logits, outcome_logits

    def predict(self, input_ids, attention_mask):
        ia_logits, genome_logits, outcome_logits = self.forward(input_ids, attention_mask)

        ia_preds = torch.argmax(ia_logits, dim=1)
        genome_preds = torch.argmax(genome_logits, dim=1)
        outcome_preds = torch.argmax(outcome_logits, dim=1)

        return ia_preds, genome_preds, outcome_preds

In [ ]:
def predict_outcomes(input_file, model_file, output_file, progressbar, progress_label):
    #maps outcomeids to outcome
    #input is programdescriptions file, .pt file
    #outputs predictions

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    #loading data from input file
    if input_file.lower().endswith('.xlsx') or input_file.lower().endswith('.xls'):
        df = pd.read_excel(input_file)
    else:
        df = pd.read_csv(input_file)
    #pre-processing programdescriptions
    df['original_programdescription'] = df['programdescription'].copy()
    df['programdescription'] = df['programdescription'].apply(
        lambda x: re.sub(r"[^A-Za-z0-9 :.,'-]+", "", str(x))
    )

    #map outcomes from input file
    outcome_dict = {}
    if 'outcomeid' in df.columns and 'outcome' in df.columns:
        outcome_mapping_df = df[['outcomeid', 'outcome']].drop_duplicates()
        outcome_dict = dict(zip(outcome_mapping_df['outcomeid'], outcome_mapping_df['outcome']))

    #load model
    #prevent errors
    from torch.serialization import add_safe_globals
    add_safe_globals([HierarchicalBertClassifier])
    try:
        checkpoint= torch.load(model_file, map_location=device, weights_only=False)
    except:
        try:
            checkpoint= torch.load(model_file, map_location=device)
        except:
            checkpoint= torch.load(
                model_file,
                map_location=device,
                pickle_module =torch.serialization.pickle_module,
                weights_only=False
            )

    model= checkpoint['model']
    model.to(device)
    model.eval()

    #get encoders
    outcome_encoder = checkpoint.get('outcome_encoder')
    genome_encoder = checkpoint.get('genome_encoder')
    impactarea_encoder = checkpoint.get('impactarea_encoder')
    outcome_mapping = checkpoint.get('outcome_mapping')

    #initalizing tokenizer
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    #processing programdescriptions
    descriptions = df['programdescription'].fillna("").astype(str).tolist()

    encodings = tokenizer.batch_encode_plus(
        descriptions,
        add_special_tokens=True,
        max_length=240,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)

    #make predictions in batches
    batch_size = 16
    total = len(input_ids)
    total_batches = (total + batch_size - 1) // batch_size

    all_ia_preds= []
    all_genome_preds= []
    #top 3 preds
    all_outcome_pred1, all_outcome_pred2, all_outcome_pred3= [],[],[]

    all_ia_conf = []
    all_genome_conf = []
    #top 3 conf
    all_outcome_conf1, all_outcome_conf2, all_outcome_conf3= [],[],[]
    
    all_ia_confmargin = []
    all_genome_confmargin = []
    all_outcome_confmargin1, all_outcome_confmargin2, all_outcome_confmargin3 = [],[],[]

    all_ia_distribution_entropy, all_genome_distribution_entropy, all_outcome_distribution_entropy = [],[],[]

    for i in range(0, total, batch_size):
        batch_input_ids = input_ids[i:i+batch_size]
        batch_attention_mask = attention_mask[i:i+batch_size]

        with torch.no_grad():
            ia_logits, genome_logits, outcome_logits = model(
                batch_input_ids, 
                batch_attention_mask)

            #convert logits to probabilities    
            ia_probs = F.softmax(ia_logits, dim=1)
            genome_probs = F.softmax(genome_logits, dim=1)
            outcome_probs = F.softmax(outcome_logits, dim=1)
            
            #get predictions
            ia_preds = torch.argmax(ia_probs, dim=1)
            genome_preds = torch.argmax(genome_probs, dim=1)
            #outcome_preds = torch.argmax(outcome_probs, dim=1)
            #get top 3 outcome predictions and confidence (4 for conf margin calc)
            outcome_top3val, outcome_top3idx = torch.topk(outcome_probs, k=4, dim=1)

            all_ia_preds.extend(ia_preds.cpu().numpy())
            all_genome_preds.extend(genome_preds.cpu().numpy())

            #calculate confidence metrics for each prediction
            for j in range(len(batch_input_ids)):
                #confidence
                #probability of predicted class
                ia_pred_idx = ia_preds[j].item() #get index of max probability 
                genome_pred_idx = genome_preds[j].item() 

                #top 3 outcome idxs/vals
                top1_idx=outcome_top3idx[j,0].item()
                top2_idx=outcome_top3idx[j,1].item()
                top3_idx=outcome_top3idx[j,2].item()

                #top 3 outcome pred for record
                all_outcome_pred1.append(top1_idx)
                all_outcome_pred2.append(top2_idx)
                all_outcome_pred3.append(top3_idx)

                ia_conf = ia_probs[j, ia_pred_idx].item() #get probability of index
                genome_conf = genome_probs[j, genome_pred_idx].item()

                #get top 3 outcome conf + 4 for margin
                conf1 = outcome_top3val[j,0].item()
                conf2 = outcome_top3val[j,1].item()
                conf3 = outcome_top3val[j,2].item()
                conf4 = outcome_top3val[j,3].item()
                
                #store conf 
                all_ia_conf.append(ia_conf)
                all_genome_conf.append(genome_conf)
                all_outcome_conf1.append(conf1)
                all_outcome_conf2.append(conf2)
                all_outcome_conf3.append(conf3)

                #calc conf margins
                #for ia and genome: gap between top 2 probabilities 
                sorted_ia_probs, _ = torch.sort(ia_probs[j], descending=True)
                sorted_genome_probs, _ = torch.sort(genome_probs[j], descending=True)
                #calc margins
                ia_margin = (sorted_ia_probs[0] - sorted_ia_probs[1]).item()
                genome_margin = (sorted_genome_probs[0] - sorted_genome_probs[1]).item() 
                #calc conf margins 
                #for outcome: gap between consecutive top predictions
                outcome_margin1 = conf1 - conf2
                outcome_margin2 = conf2 - conf3
                outcome_margin3 = conf3 - conf4

                #store conf margins
                all_ia_confmargin.append(ia_margin)
                all_genome_confmargin.append(genome_margin)
                all_outcome_confmargin1.append(outcome_margin1)
                all_outcome_confmargin2.append(outcome_margin2)
                all_outcome_confmargin3.append(outcome_margin3)

                #calc uncertainty/entropy
                ia_entropy = -torch.sum(ia_probs[j] * torch.log2(ia_probs[j] + 1e-10)).item()
                genome_entropy = -torch.sum(genome_probs[j] * torch.log2(genome_probs[j] + 1e-10)).item()
                outcome_entropy = -torch.sum(outcome_probs[j] * torch.log2(outcome_probs[j] + 1e-10)).item()
                #store distribution entropy
                all_ia_distribution_entropy.append(ia_entropy)
                all_genome_distribution_entropy.append(genome_entropy)
                all_outcome_distribution_entropy.append(outcome_entropy)


        progress = int((i + batch_size) / total * 100)
        progressbar["value"] = min(progress, 100)
        progress_label.config(text=f"Processing batch {i//batch_size + 1}/{total_batches}")
        progressbar.update()
        progress_label.update()
        
    #df with only programdescription and predictions
    result_df = pd.DataFrame()
    #result_df['programdescription'] = df['programdescription']
    result_df['original_programdescription'] = df['original_programdescription']
    
    #convert predictions to labels
    if outcome_encoder:
        result_df['predicted_impact_area']=impactarea_encoder.inverse_transform(all_ia_preds)
        result_df['predicted_genome']=genome_encoder.inverse_transform(all_genome_preds)
        #result_df['predicted_outcomeid']=outcome_encoder.inverse_transform(all_outcome_preds)
        result_df['predicted_outcomeid1']=outcome_encoder.inverse_transform(all_outcome_pred1)
        result_df['predicted_outcomeid2']=outcome_encoder.inverse_transform(all_outcome_pred2)
        result_df['predicted_outcomeid3']=outcome_encoder.inverse_transform(all_outcome_pred3)

        #map outcomeid to outcome
        if outcome_mapping:
            #result_df['predicted_outcome'] = result_df['predicted_outcomeid'].map(outcome_mapping)
            result_df['predicted_outcome1'] = result_df['predicted_outcomeid1'].map(outcome_mapping) 
            result_df['predicted_outcome2'] = result_df['predicted_outcomeid2'].map(outcome_mapping) 
            result_df['predicted_outcome3'] = result_df['predicted_outcomeid3'].map(outcome_mapping) 
    else:
        #result_df['predicted_outcomeid'] = all_outcome_preds
        result_df['predicted_outcomeid1'] = all_outcome_pred1
        result_df['predicted_outcomeid2'] = all_outcome_pred2
        result_df['predicted_outcomeid3'] = all_outcome_pred3
        result_df['predicted_genome_id'] = all_genome_preds
        result_df['predicted_impact_area_id'] = all_ia_preds

    #add confidence metrics
    result_df['impact_area_confidence'] = all_ia_conf
    result_df['genome_confidence'] = all_genome_conf
    result_df['outcome_confidence1'] = all_outcome_conf1
    result_df['outcome_confidence2'] = all_outcome_conf2
    result_df['outcome_confidence3'] = all_outcome_conf3

    #add confidence margin metrics
    result_df['impact_area_confidence_margin'] = all_ia_confmargin
    result_df['genome_confidence_margin'] = all_genome_confmargin
    result_df['outcome_confidence_margin1'] = all_outcome_confmargin1
    result_df['outcome_confidence_margin2'] = all_outcome_confmargin2
    result_df['outcome_confidence_margin3'] = all_outcome_confmargin3

    #calc max entropy
    num_ia_classes = len(impactarea_encoder.classes_) 
    num_genome_classes = len(genome_encoder.classes_) 
    num_outcome_classes = len(outcome_encoder.classes_)

    max_ia_entropy = np.log2(num_ia_classes)
    max_genome_entropy = np.log2(num_genome_classes)
    max_outcome_entropy = np.log2(num_outcome_classes)

    #calc inverse standardized entropy (higher vals = more certainty)
    ia_certainty = [1 - (entropy / max_ia_entropy) for entropy in all_ia_distribution_entropy]
    genome_certainty = [1 - (entropy / max_genome_entropy) for entropy in all_genome_distribution_entropy]
    outcome_certainty = [1 - (entropy / max_outcome_entropy) for entropy in all_outcome_distribution_entropy]

    #add certainty metrics
    result_df['impact_area_certainty'] = ia_certainty
    result_df['genome_certainty'] = genome_certainty
    result_df['outcome_certainty'] = outcome_certainty

    def calc_comp_score(confidence, margin, certainty, weights=[0.6, 0.15, 0.25]):
        #weight metrics - linear combination
        composite_score=(
            weights[0] * confidence +
            weights[1] * margin + 
            weights[2] * certainty
        )
        return composite_score

    #calc composite for each pred
    ia_comp= [calc_comp_score(conf, margin, cert) 
            for conf, margin, cert in zip(all_ia_conf, all_ia_confmargin, ia_certainty)]

    genome_comp= [calc_comp_score(conf, margin, cert) 
                for conf, margin, cert in zip(all_genome_conf, all_genome_confmargin, genome_certainty)]

    outcome_comp1= [calc_comp_score(conf, margin, cert) 
                    for conf, margin, cert in zip(all_outcome_conf1, all_outcome_confmargin1, outcome_certainty)]

    outcome_comp2= [calc_comp_score(conf, margin, cert) 
                    for conf, margin, cert in zip(all_outcome_conf2, all_outcome_confmargin2, outcome_certainty)]

    outcome_comp3= [calc_comp_score(conf, margin, cert) 
                    for conf, margin, cert in zip(all_outcome_conf3, all_outcome_confmargin3, outcome_certainty)]
    #add to result df
    result_df['impact_area_composite']= ia_comp
    result_df['genome_composite']= genome_comp
    result_df['outcome_composite1']= outcome_comp1
    result_df['outcome_composite2']= outcome_comp2
    result_df['outcome_composite3']= outcome_comp3

    result_df.to_csv(output_file, index=False)
    return result_df


In [17]:
def run_prediction_gui():
    def browse_input():
        #allow csv or excel files as input
        filename = filedialog.askopenfilename(filetypes=[("Data Files", "*.csv *.xlsx *.xls")])
        input_entry.delete(0, tk.END)
        input_entry.insert(0, filename)

    def browse_model():
        #model is .pt file
        filename = filedialog.askopenfilename(filetypes=[("PyTorch Model", "*.pt")])
        model_entry.delete(0, tk.END)
        model_entry.insert(0, filename)

    def browse_output():
        #only allow csv
        filename = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV files", "*.csv")])
        output_entry.delete(0, tk.END)
        output_entry.insert(0, filename)

    def run_prediction():
        #get input, model, and output files
        input_file = input_entry.get()
        model_file = model_entry.get()
        output_file = output_entry.get()
        #if incomplete, prompt user
        if not all([input_file, model_file, output_file]):
            messagebox.showwarning("Missing Info", "Please provide all files.")
            return
        #progress bar
        progress["value"] = 0
        progress_label.config(text="Starting prediction...")
        #run thread for prediction task to prevent freezing GUI
        threading.Thread(
            target=run_predict_outcomes_thread,
            args=(input_file, model_file, output_file)
        ).start()

    def run_predict_outcomes_thread(input_file, model_file, output_file):
        try:
            result_df = predict_outcomes(
                input_file, model_file, output_file, progress, progress_label
            )
            messagebox.showinfo("Complete", f"Predictions saved to:\n{output_file}")
        except Exception as e:
            messagebox.showerror("Error", str(e))

    root = tk.Tk()
    root.title("Outcome Predictor")

    #layout
    #input file
    tk.Label(root, text="Input File:").grid(row=0, column=0, sticky="e")
    input_entry = tk.Entry(root, width=50)
    input_entry.grid(row=0, column=1, padx=5)
    tk.Button(root, text="Browse", command=browse_input).grid(row=0, column=2)
    #model file
    tk.Label(root, text="Model File:").grid(row=1, column=0, sticky="e")
    model_entry = tk.Entry(root, width=50)
    model_entry.grid(row=1, column=1, padx=5)
    tk.Button(root, text="Browse", command=browse_model).grid(row=1, column=2)
    #output file
    tk.Label(root, text="Output File:").grid(row=2, column=0, sticky="e")
    output_entry = tk.Entry(root, width=50)
    output_entry.grid(row=2, column=1, padx=5)
    tk.Button(root, text="Save As", command=browse_output).grid(row=2, column=2)
    #get prediction
    tk.Button(root, text="Get Predictions", command=run_prediction).grid(row=3, column=1, pady=10)

    #progress bar
    progress = ttk.Progressbar(root, orient="horizontal", length=400, mode="determinate")
    progress.grid(row=4, column=0, columnspan=3, pady=5)
    progress_label = tk.Label(root, text="")
    progress_label.grid(row=5, column=0, columnspan=3)

    root.mainloop()


In [18]:
run_prediction_gui()


c:\Users\antek\anaconda3\envs\new_env\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
